In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

cars = pd.read_csv('../data/processed/cars_unified.csv')
cd   = pd.read_csv('../data/processed/cars_cd.csv')

print("Cars unified:", cars.shape)
print("\nVehicle classes:", sorted(cars['vehicle_class'].unique()))
print("\nFuel types:", sorted(cars['fuel_type'].unique()))
print("\nMPG stats:")
print(cars[['city_mpg','highway_mpg','combined_mpg']].describe().round(1))

In [2]:
# ── Cell 2: MPG distribution + engine displacement ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MPG distribution
axes[0].hist(cars['city_mpg'], bins=40, alpha=0.7, 
             color='#e10600', label='City', edgecolor='none')
axes[0].hist(cars['highway_mpg'], bins=40, alpha=0.7,
             color='#1e1e2e', label='Highway', edgecolor='none')
axes[0].set_title('MPG distribution — city vs highway')
axes[0].set_xlabel('MPG')
axes[0].set_ylabel('Count')
axes[0].legend()

# MPG vs engine displacement scatter
axes[1].scatter(cars['engine_displacement_l'], cars['combined_mpg'],
                alpha=0.2, color='#e10600', edgecolors='none', s=10)
axes[1].set_title('Combined MPG vs engine displacement')
axes[1].set_xlabel('Engine displacement (L)')
axes[1].set_ylabel('Combined MPG')

# Trend line
mask = cars[['engine_displacement_l','combined_mpg']].dropna()
z = np.polyfit(mask['engine_displacement_l'], mask['combined_mpg'], 1)
p = np.poly1d(z)
x_line = np.linspace(mask['engine_displacement_l'].min(),
                     mask['engine_displacement_l'].max(), 100)
axes[1].plot(x_line, p(x_line), color='#1e1e2e', linewidth=2)

corr = mask['engine_displacement_l'].corr(mask['combined_mpg'])
axes[1].set_title(f'Combined MPG vs engine displacement (r = {corr:.2f})')

plt.tight_layout()
plt.savefig('../data/processed/plot_mpg_dist.png', dpi=150)
plt.show()
print(f"Correlation MPG vs displacement: {corr:.3f}")

In [3]:
# ── Cell 3: MPG by vehicle class + Cd vs MPG ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Median combined MPG by vehicle class
class_mpg = (cars.groupby('vehicle_class')['combined_mpg']
             .median().sort_values(ascending=False))
class_mpg.index = class_mpg.index.str.replace(' Vehicle', '').str.replace(' Cars', '')

axes[0].barh(class_mpg.index, class_mpg.values,
             color='#e10600', edgecolor='none')
axes[0].set_title('Median combined MPG by vehicle class')
axes[0].set_xlabel('Combined MPG')
axes[0].axvline(class_mpg.mean(), color='#1e1e2e',
                linestyle='--', label='Mean')
axes[0].legend()

# Cd vs highway MPG (aero efficiency insight)
cd_cars = cars[cars['cd'].notna()]
axes[1].scatter(cd_cars['cd'], cd_cars['highway_mpg'],
                alpha=0.5, color='#e10600', edgecolors='none', s=40)

# Label notable cars
for _, row in cd_cars.drop_duplicates('cd').iterrows():
    if row['cd'] < 0.22 or row['cd'] > 0.45:
        axes[1].annotate(f"{row['make']} {row['model']}",
                        (row['cd'], row['highway_mpg']),
                        fontsize=7, alpha=0.8,
                        xytext=(5, 5), textcoords='offset points')

z = np.polyfit(cd_cars['cd'], cd_cars['highway_mpg'], 1)
p = np.poly1d(z)
x_line = np.linspace(cd_cars['cd'].min(), cd_cars['cd'].max(), 100)
axes[1].plot(x_line, p(x_line), color='#1e1e2e', linewidth=2)

corr = cd_cars['cd'].corr(cd_cars['highway_mpg'])
axes[1].set_title(f'Drag coefficient (Cd) vs highway MPG (r = {corr:.2f})')
axes[1].set_xlabel('Drag coefficient (Cd)')
axes[1].set_ylabel('Highway MPG')

plt.tight_layout()
plt.savefig('../data/processed/plot_cd_mpg.png', dpi=150)
plt.show()
print(f"Correlation Cd vs highway MPG: {corr:.3f}")

In [4]:
# ── Cell 4: MPG trend over years + fuel type ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Average combined MPG per year (improvement over time)
yearly_mpg = (cars.groupby('year')['combined_mpg']
              .mean().reset_index())
axes[0].plot(yearly_mpg['year'], yearly_mpg['combined_mpg'],
             color='#e10600', linewidth=2.5, marker='o', markersize=4)
axes[0].fill_between(yearly_mpg['year'], yearly_mpg['combined_mpg'],
                     alpha=0.15, color='#e10600')
axes[0].set_title('Average combined MPG over time (2010–2026)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Combined MPG')
axes[0].set_ylim(yearly_mpg['combined_mpg'].min() - 2,
                 yearly_mpg['combined_mpg'].max() + 2)

# MPG by fuel type
fuel_mpg = (cars.groupby('fuel_type')['combined_mpg']
            .median().sort_values(ascending=False))
fuel_mpg.index = fuel_mpg.index.str.replace(' or Electricity', '+EV')
fuel_mpg.index = fuel_mpg.index.str.replace(' and Electricity', '+EV')

axes[1].bar(range(len(fuel_mpg)), fuel_mpg.values,
            color='#1e1e2e', edgecolor='none')
axes[1].set_xticks(range(len(fuel_mpg)))
axes[1].set_xticklabels(fuel_mpg.index, rotation=30, ha='right', fontsize=9)
axes[1].set_title('Median combined MPG by fuel type')
axes[1].set_ylabel('Combined MPG')

plt.tight_layout()
plt.savefig('../data/processed/plot_mpg_trends.png', dpi=150)
plt.show()